In [12]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Desktop/comparision/Not_Source_updated.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.8603, 0.0024),
    "z": (30.4519, 0.0036),
    "y": (28.2513, 0.0029)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# I-band non-detections ONLY
# ============================================================
I_LIMIT = 27.46 #27.46

i_nondet = (~i_detected) | (~np.isfinite(i_mag))

# apply limit ONLY to true non-detections
i_mag_lim = i_mag.copy()
i_err_lim = i_err.copy()

i_mag_lim[i_nondet] = I_LIMIT
i_err_lim[i_nondet] = np.inf   # important

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag_lim - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = y_snr > 4.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

# dropout is LOGICAL, not photometric
cut_i_dropout  = (~i_detected) | (i_snr < 2.0)

cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"Selected LBG candidates: {len(lbg_idx)}")

# ============================================================
# Build output catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag_lim[lbg_idx],
    "I_err": i_err_lim[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, "
        f"Z_mag={row.Z_mag:.2f}, "
        f"Y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 414747
Selected LBG candidates: 577
Final LBG candidates: 135
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final candidates:
RA=357.278631, DEC=-31.660724, I_SNR=0.00, I_mag=27.46, Z_mag=25.42, Y_mag=23.89
RA=357.430310, DEC=-31.651233, I_SNR=1.98, I_mag=26.67, Z_mag=25.55, Y_mag=24.62
RA=356.766797, DEC=-31.636807, I_SNR=0.64, I_mag=27.88, Z_mag=25.43, Y_mag=24.44
RA=356.715290, DEC=-31.636123, I_SNR=0.00, I_mag=27.46, Z_mag=25.55, Y_mag=24.52
RA=357.459845, DEC=-31.629389, I_SNR=0.00, I_mag=27.46, Z_mag=25.54, Y_mag=24.72
RA=356.449227, DEC=-31.624836, I_SNR=0.92, I_mag=27.50, Z_mag=25.41, Y_mag=24.50
RA=356.946582, DEC=-31.611597, I_SNR=0.00, I_mag=27.46, Z_mag=25.72, Y_mag=24.72
RA=356.860591, DEC=-31.590200, I_SNR=1.87, I_mag=26.70, Z_mag=24.11, Y_mag=23.26
RA=357.184588, DEC=-31.587619, I_SNR=0.01, I_mag=129.86, Z_mag=24.86, Y_mag=23.86
RA=356.837914, DEC=-31.585759, I_SNR=0.00, I_mag=27.46, Z_mag=25.55, Y_mag=24.61
RA=357.260189, DEC=-31

In [8]:
# ============================================================
# Check a specific source
# ============================================================
target_ra = 358.193029
target_dec = -30.9429872

# Find closest Y-source
target_coord = SkyCoord(target_ra, target_dec, unit="deg")
sep = target_coord.separation(y_coords)
target_idx = np.argmin(sep.arcsec)

print(f"Closest Y-source index: {target_idx}, separation: {sep[target_idx].arcsec:.3f} arcsec")

# Print relevant properties
print("Photometry and detection flags for this source:")
print(f"Y_mag = {y_mag[target_idx]:.2f}, Y_err = {y_err[target_idx]:.2f}, Y_SNR = {y_snr[target_idx]:.2f}")
print(f"Z_mag = {z_mag[target_idx]:.2f}, Z_err = {z_err[target_idx]:.2f}, Z_SNR = {z_snr[target_idx]:.2f}, Z_detected = {z_detected[target_idx]}")
print(f"I_mag = {i_mag_lim[target_idx]:.2f}, I_err = {i_err_lim[target_idx]:.2f}, I_SNR = {i_snr[target_idx]:.2f}, I_detected = {i_detected[target_idx]}")
print(f"Colors: Z-Y = {color_zy[target_idx]:.2f}, I-Z = {color_iz[target_idx]:.2f}")

# Check all cuts
cuts = {
    "cut_z_detected": cut_z_detected[target_idx],
    "cut_z_snr": cut_z_snr[target_idx],
    "cut_y_snr": cut_y_snr[target_idx],
    "cut_zy": cut_zy[target_idx],
    "cut_break": cut_break[target_idx],
    "cut_i_dropout": cut_i_dropout[target_idx],
    "cut_sig": cut_sig[target_idx],
}

for k, v in cuts.items():
    print(f"{k}: {v}")

print(f"Final selection: {final_sel[target_idx]}")

Closest Y-source index: 177606, separation: 0.014 arcsec
Photometry and detection flags for this source:
Y_mag = 24.73, Y_err = 0.24, Y_SNR = 4.59
Z_mag = 25.86, Z_err = 0.27, Z_SNR = 4.00, Z_detected = True
I_mag = 26.75, I_err = 0.56, I_SNR = 1.93, I_detected = True
Colors: Z-Y = 1.13, I-Z = 0.89
cut_z_detected: True
cut_z_snr: True
cut_y_snr: True
cut_zy: True
cut_break: False
cut_i_dropout: True
cut_sig: True
Final selection: False
